# Chapter 2.3: Memory Management for KV-Cache

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the memory fragmentation problem** in naive KV-cache management
2. **Explain the virtual memory analogy** and how PagedAttention applies it to KV-cache
3. **Implement a block-based KV-cache allocator** from scratch
4. **Demonstrate Copy-on-Write** for parallel sampling (beam search, best-of-N)
5. **Implement prefix sharing** to reuse KV-cache across requests
6. **Understand CPU offloading** for preempted request KV-cache
7. **Visualize memory fragmentation** with and without paging

## Prerequisites

- Chapter 2.1: Request Lifecycle (understanding KV-cache role)
- Chapter 2.2: Scheduler Design (preemption, admission control)
- Basic OS concepts: virtual memory, page tables, paging

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part2/chapter_2.3_kvcache_memory.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part2/chapter_2.3_kvcache_memory.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The KV-Cache Memory Problem

In autoregressive LLM inference, each generated token requires attending to **all previous tokens**. The Key and Value tensors from previous tokens are cached to avoid recomputation -- this is the **KV-cache**.

### Memory Size of KV-Cache

For a single request:

$$\text{KV\_cache\_size} = 2 \times n_{\text{layers}} \times n_{\text{KV\_heads}} \times d_{\text{head}} \times s \times \text{sizeof(dtype)}$$

Where:
- $2$ = Key and Value
- $n_{\text{layers}}$ = number of transformer layers
- $n_{\text{KV\_heads}}$ = number of KV attention heads (may be fewer with GQA)
- $d_{\text{head}}$ = dimension per head
- $s$ = current sequence length
- $\text{sizeof(dtype)}$ = 2 bytes for FP16

### The Core Challenge

The KV-cache **grows dynamically** as tokens are generated, and its final size is **unknown in advance** (depends on when EOS is hit). This makes static memory allocation fundamentally wasteful.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set, Tuple
import random
import warnings
warnings.filterwarnings('ignore')

# KV-Cache size calculator
def kv_cache_size_per_token(num_layers, num_kv_heads, head_dim, dtype_bytes=2):
    """Bytes of KV-cache per token."""
    return 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

# Popular model configurations
models = {
    'Llama-3-8B': {'layers': 32, 'kv_heads': 8, 'head_dim': 128, 'max_ctx': 8192},
    'Llama-3-70B': {'layers': 80, 'kv_heads': 8, 'head_dim': 128, 'max_ctx': 8192},
    'Mistral-7B': {'layers': 32, 'kv_heads': 8, 'head_dim': 128, 'max_ctx': 32768},
    'GPT-4 (est.)': {'layers': 120, 'kv_heads': 16, 'head_dim': 128, 'max_ctx': 128000},
}

print("=== KV-Cache Memory Requirements ===")
print(f"{'Model':<16} {'Per Token':<12} {'1K tokens':<12} {'4K tokens':<12} {'Max Context':<14} {'Max KV-Cache':<14}")
print("-" * 80)

for name, cfg in models.items():
    per_token = kv_cache_size_per_token(cfg['layers'], cfg['kv_heads'], cfg['head_dim'])
    kv_1k = per_token * 1024
    kv_4k = per_token * 4096
    kv_max = per_token * cfg['max_ctx']
    
    print(f"{name:<16} {per_token/1024:>8.1f} KB   {kv_1k/1024/1024:>8.1f} MB   "
          f"{kv_4k/1024/1024:>8.1f} MB   {cfg['max_ctx']:>10,}     {kv_max/1024/1024/1024:>10.2f} GB")

In [ ]:
## Figure: Contiguous vs Paged KV-Cache Allocation
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

GREEN, RED, BLUE, ORANGE = '#27AE60', '#E74C3C', '#4A90D9', '#F39C12'

# TOP: Contiguous allocation (wasteful)
ax = axes[0]
ax.set_xlim(0, 20)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('Contiguous Allocation: Internal + External Fragmentation', fontsize=13, fontweight='bold', color=RED)

# 3 requests with reserved (max) vs actual usage
allocs = [
    ('Req A', 0, 5, 2, BLUE),     # reserved 5, used 2
    ('Req B', 5, 7, 7, GREEN),     # reserved 7, used 7 (finished)
    ('Req C', 12, 4, 1.5, ORANGE), # reserved 4, used 1.5
]

for name, start, reserved, used, color in allocs:
    # Reserved block
    ax.add_patch(patches.Rectangle((start, 1), reserved, 1, facecolor=color, alpha=0.2,
                 edgecolor=color, linewidth=2, linestyle='--'))
    # Actually used
    ax.add_patch(patches.Rectangle((start, 1), used, 1, facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(start + used/2, 1.5, f'{name}\n(used)', ha='center', fontsize=8, fontweight='bold', color='white')
    if reserved > used:
        ax.text(start + (reserved + used)/2, 1.5, 'WASTED', ha='center', fontsize=7, color=RED, fontweight='bold')

# Free space
ax.add_patch(patches.Rectangle((16, 1), 3.5, 1, facecolor='lightgray', alpha=0.3, edgecolor='gray'))
ax.text(17.75, 1.5, 'Free', ha='center', fontsize=9, fontweight='bold', color='gray')

ax.text(10, 0.3, 'Internal fragmentation (reserved > used) + External fragmentation (non-contiguous free space)',
        ha='center', fontsize=9, color=RED, fontweight='bold')

# BOTTOM: Paged allocation (efficient)
ax = axes[1]
ax.set_xlim(0, 20)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('Paged Allocation (PagedAttention): No Fragmentation', fontsize=13, fontweight='bold', color=GREEN)

# Blocks allocated on-demand, non-contiguous
block_map = [
    (0, BLUE), (1, ORANGE), (2, BLUE), (3, GREEN), (4, GREEN), 
    (5, ORANGE), (6, None), (7, GREEN), (8, None), (9, BLUE),
    (10, GREEN), (11, None), (12, None), (13, GREEN), (14, None),
    (15, ORANGE), (16, None), (17, None), (18, None), (19, None)
]

for i, (idx, color) in enumerate(block_map):
    x = i * 1.0
    if color:
        ax.add_patch(patches.Rectangle((x, 1), 0.9, 1, facecolor=color, alpha=0.7, edgecolor='black'))
        label = {'#4A90D9': 'A', '#27AE60': 'B', '#F39C12': 'C'}.get(color, '?')
        ax.text(x + 0.45, 1.5, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    else:
        ax.add_patch(patches.Rectangle((x, 1), 0.9, 1, facecolor='lightgray', alpha=0.2, edgecolor='gray'))

ax.text(10, 0.3, 'Blocks allocated on-demand, non-contiguous. Only actual tokens consume memory.',
        ha='center', fontsize=9, color=GREEN, fontweight='bold')

plt.suptitle('PagedAttention Motivation: Eliminating Memory Waste', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("PagedAttention applies OS virtual memory concepts to KV-cache management.")
print("Result: near-zero memory waste and no fragmentation issues.")

## 2. The Memory Fragmentation Problem (Without Paging)

### Naive Approach: Contiguous Allocation

The simplest approach: allocate a contiguous block of GPU memory for each request's KV-cache, sized for `max_tokens`.

**Problems with this approach:**

1. **Internal fragmentation**: A request that generates only 50 tokens but reserved space for 2048 wastes 97.5% of its allocation
2. **External fragmentation**: After requests complete, the freed memory may be in non-contiguous chunks, unable to serve a large new request
3. **Over-provisioning**: Must reserve for worst case (max_tokens), drastically reducing batch size

Let's visualize this problem.

In [ ]:
def visualize_fragmentation():
    """Visualize memory fragmentation with contiguous allocation."""
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), gridspec_kw={'hspace': 0.4})
    
    total_memory = 100  # Abstract memory units
    
    # State 1: Initial allocation (3 requests, each reserves max_tokens)
    ax = axes[0]
    ax.set_title('Time 0: Three requests allocated (max_tokens reservation)', fontweight='bold')
    ax.set_xlim(0, total_memory)
    ax.set_ylim(0, 2)
    ax.barh(1, 30, left=0, height=0.8, color='#3498db', alpha=0.8, edgecolor='black', label='Req A (reserved 30)')
    ax.barh(1, 35, left=30, height=0.8, color='#e74c3c', alpha=0.8, edgecolor='black', label='Req B (reserved 35)')
    ax.barh(1, 25, left=65, height=0.8, color='#2ecc71', alpha=0.8, edgecolor='black', label='Req C (reserved 25)')
    ax.barh(1, 10, left=90, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    # Show actual usage within each allocation
    ax.barh(0.3, 10, left=0, height=0.4, color='#3498db', alpha=1.0, edgecolor='black')
    ax.barh(0.3, 5, left=30, height=0.4, color='#e74c3c', alpha=1.0, edgecolor='black')
    ax.barh(0.3, 8, left=65, height=0.4, color='#2ecc71', alpha=1.0, edgecolor='black')
    ax.text(50, 0.3, 'Actual usage', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(50, 1, 'Reserved space', ha='center', va='center', fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    ax.text(95, 1, 'Free\n10 units', ha='center', va='center', fontsize=8)
    ax.set_yticks([])
    
    # State 2: Request B completes early, leaving a gap
    ax = axes[1]
    ax.set_title('Time 1: Request B completes (used only 15 of 35 reserved)', fontweight='bold')
    ax.set_xlim(0, total_memory)
    ax.set_ylim(0, 2)
    ax.barh(1, 30, left=0, height=0.8, color='#3498db', alpha=0.8, edgecolor='black')
    ax.barh(1, 35, left=30, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    ax.barh(1, 25, left=65, height=0.8, color='#2ecc71', alpha=0.8, edgecolor='black')
    ax.barh(1, 10, left=90, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    ax.text(47.5, 1, 'FREE (35 units)', ha='center', va='center', fontsize=10, fontweight='bold', color='gray')
    ax.text(95, 1, 'Free\n10', ha='center', va='center', fontsize=8)
    ax.annotate('Internal fragmentation:\nB used only 15/35 reserved', 
                xy=(47.5, 0.5), fontsize=9, color='red', fontweight='bold', ha='center')
    ax.set_yticks([])
    
    # State 3: New request D needs 40 units - doesn't fit!
    ax = axes[2]
    ax.set_title('Time 2: Request D arrives (needs 40 units) -- CANNOT FIT!', fontweight='bold', color='red')
    ax.set_xlim(0, total_memory)
    ax.set_ylim(0, 2)
    ax.barh(1, 30, left=0, height=0.8, color='#3498db', alpha=0.8, edgecolor='black')
    ax.barh(1, 35, left=30, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    ax.barh(1, 25, left=65, height=0.8, color='#2ecc71', alpha=0.8, edgecolor='black')
    ax.barh(1, 10, left=90, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    # Show D needs 40 contiguous units
    ax.barh(0.3, 40, left=30, height=0.4, color='#f39c12', alpha=0.5, edgecolor='red', linewidth=2, linestyle='--')
    ax.text(50, 0.3, 'Req D needs 40 units', ha='center', va='center', fontsize=9, color='red', fontweight='bold')
    ax.annotate('External fragmentation!\n45 units free but largest contiguous = 35', 
                xy=(50, 1.7), fontsize=10, color='red', fontweight='bold', ha='center')
    ax.set_yticks([])
    
    # State 4: With paging, D can fit!
    ax = axes[3]
    ax.set_title('With Paging: Request D fits using non-contiguous blocks!', fontweight='bold', color='green')
    ax.set_xlim(0, total_memory)
    ax.set_ylim(0, 2)
    ax.barh(1, 20, left=0, height=0.8, color='#3498db', alpha=0.8, edgecolor='black')
    ax.barh(1, 15, left=20, height=0.8, color='#f39c12', alpha=0.8, edgecolor='black')  # D part 1
    ax.barh(1, 5, left=35, height=0.8, color='#f39c12', alpha=0.8, edgecolor='black')   # D part 2
    ax.barh(1, 20, left=40, height=0.8, color='#f39c12', alpha=0.8, edgecolor='black')  # D part 3
    ax.barh(1, 15, left=60, height=0.8, color='#2ecc71', alpha=0.8, edgecolor='black')
    ax.barh(1, 25, left=75, height=0.8, color='lightgray', alpha=0.3, edgecolor='black', hatch='//')
    ax.text(15, 0.3, 'A (only uses\nactual tokens)', ha='center', fontsize=8, color='#3498db', fontweight='bold')
    ax.text(40, 0.3, 'D (non-contiguous blocks)', ha='center', fontsize=8, color='#f39c12', fontweight='bold')
    ax.text(67.5, 0.3, 'C (actual)', ha='center', fontsize=8, color='#2ecc71', fontweight='bold')
    ax.set_yticks([])
    ax.set_xlabel('GPU Memory (abstract units)')
    
    plt.suptitle('Memory Fragmentation: Contiguous vs Paged Allocation', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

visualize_fragmentation()

## 3. Virtual Memory Analogy: Page Tables for KV-Cache

The insight of **PagedAttention** (vLLM) is to apply the same solution that operating systems use for RAM:

### OS Virtual Memory
- Physical RAM is divided into fixed-size **pages** (e.g., 4KB)
- Each process has a **page table** mapping virtual addresses to physical pages
- Pages can be **non-contiguous** in physical memory
- Pages can be **swapped** to disk when memory is low

### KV-Cache Paging (PagedAttention)
- GPU memory is divided into fixed-size **KV blocks** (e.g., 16 tokens per block)
- Each request has a **block table** mapping logical blocks to physical blocks
- Blocks can be **non-contiguous** in GPU memory
- Blocks can be **swapped** to CPU memory when preempted

| OS Concept | KV-Cache Analog |
|-----------|----------------|
| Virtual address space | Request's logical KV-cache |
| Physical page | KV block in GPU memory |
| Page table | Block table |
| Page fault | Allocate new block |
| Page eviction | Preemption (swap/recompute) |
| Shared pages | Prefix sharing / CoW |

In [ ]:
## Figure: Block Table -- Logical to Physical Block Mapping
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 20)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Block Table: Mapping Logical Blocks to Physical GPU Memory', fontsize=15, fontweight='bold')

BLUE, GREEN, ORANGE = '#4A90D9', '#27AE60', '#F39C12'

# Request A - Logical blocks
ax.text(3, 11, 'Request A (Logical View)', fontsize=12, fontweight='bold', color=BLUE, ha='center')
for i in range(4):
    ax.add_patch(patches.FancyBboxPatch((1+i*1.5, 9.5), 1.2, 1, boxstyle="round,pad=0.08",
                 facecolor=BLUE, alpha=0.7, edgecolor='black'))
    ax.text(1.6+i*1.5, 10, f'L{i}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(1.6+i*1.5, 9.7, '16 tok', ha='center', fontsize=6, color='white')

# Block Table A
ax.text(9, 11, 'Block Table A', fontsize=11, fontweight='bold', ha='center')
table_a = [(0, 7), (1, 2), (2, 12), (3, 5)]
for i, (logical, physical) in enumerate(table_a):
    ax.add_patch(patches.Rectangle((8, 10.2 - i*0.6), 2, 0.5, facecolor='lightyellow', edgecolor='black'))
    ax.text(9, 10.45 - i*0.6, f'L{logical} -> GPU[{physical}]', ha='center', fontsize=8)
    # Arrow from logical to table
    ax.annotate('', xy=(8, 10.45 - i*0.6), xytext=(1+logical*1.5+1.2, 10),
                arrowprops=dict(arrowstyle='->', color=BLUE, alpha=0.3, lw=1))

# Request B - Logical blocks
ax.text(3, 7, 'Request B (Logical View)', fontsize=12, fontweight='bold', color=GREEN, ha='center')
for i in range(3):
    ax.add_patch(patches.FancyBboxPatch((1+i*1.5, 5.5), 1.2, 1, boxstyle="round,pad=0.08",
                 facecolor=GREEN, alpha=0.7, edgecolor='black'))
    ax.text(1.6+i*1.5, 6, f'L{i}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Block Table B
ax.text(9, 7, 'Block Table B', fontsize=11, fontweight='bold', ha='center')
table_b = [(0, 1), (1, 4), (2, 14)]
for i, (logical, physical) in enumerate(table_b):
    ax.add_patch(patches.Rectangle((8, 6.2 - i*0.6), 2, 0.5, facecolor='#EAFAF1', edgecolor='black'))
    ax.text(9, 6.45 - i*0.6, f'L{logical} -> GPU[{physical}]', ha='center', fontsize=8)

# Physical GPU Memory
ax.text(15, 11, 'Physical GPU Memory (KV Blocks)', fontsize=12, fontweight='bold', ha='center')
gpu_blocks = {
    0: ('free', 'lightgray'), 1: ('B[0]', GREEN), 2: ('A[1]', BLUE), 3: ('free', 'lightgray'),
    4: ('B[1]', GREEN), 5: ('A[3]', BLUE), 6: ('free', 'lightgray'), 7: ('A[0]', BLUE),
    8: ('free', 'lightgray'), 9: ('free', 'lightgray'), 10: ('free', 'lightgray'), 11: ('free', 'lightgray'),
    12: ('A[2]', BLUE), 13: ('free', 'lightgray'), 14: ('B[2]', GREEN), 15: ('free', 'lightgray'),
}

cols = 4
for bid, (label, color) in gpu_blocks.items():
    row = bid // cols
    col = bid % cols
    x = 12 + col * 2
    y = 9.5 - row * 1.2
    alpha = 0.7 if color != 'lightgray' else 0.2
    ax.add_patch(patches.Rectangle((x, y), 1.8, 0.9, facecolor=color, alpha=alpha, edgecolor='black'))
    ax.text(x + 0.9, y + 0.45, f'[{bid}] {label}', ha='center', va='center', fontsize=7, fontweight='bold')

# Arrows from table to physical
for logical, physical in table_a:
    row = physical // cols
    col = physical % cols
    px = 12 + col * 2 + 0.9
    py = 9.95 - row * 1.2
    ax.annotate('', xy=(px, py), xytext=(10, 10.45 - logical*0.6),
                arrowprops=dict(arrowstyle='->', color=BLUE, alpha=0.4, lw=1,
                               connectionstyle='arc3,rad=0.1'))

plt.tight_layout()
plt.show()
print("Each request has its own block table mapping logical blocks to non-contiguous physical blocks.")
print("This is identical to how OS page tables map virtual pages to physical frames.")

In [ ]:
# Visualize the page table analogy

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: OS Virtual Memory
ax = axes[0]
ax.set_title('OS Virtual Memory', fontsize=13, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Virtual address space
for i in range(5):
    color = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#3498db'][i]
    ax.add_patch(plt.Rectangle((0.5, 8-i*1.2), 2, 0.8, facecolor=color, alpha=0.6, edgecolor='black'))
    ax.text(1.5, 8.4-i*1.2, f'VPage {i}', ha='center', va='center', fontsize=9)

ax.text(1.5, 9.2, 'Virtual Pages\n(contiguous)', ha='center', fontweight='bold', fontsize=10)

# Page table
ax.text(4.5, 9.2, 'Page Table', ha='center', fontweight='bold', fontsize=10)
for i in range(5):
    phys = [3, 0, 4, 1, 2][i]  # mapping
    ax.add_patch(plt.Rectangle((3.5, 8-i*1.2), 2, 0.8, facecolor='lightyellow', edgecolor='black'))
    ax.text(4.5, 8.4-i*1.2, f'V{i} -> P{phys}', ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(2.5, 8.4-i*1.2), xytext=(3.5, 8.4-i*1.2),
                arrowprops=dict(arrowstyle='->', color='gray'))

# Physical memory (non-contiguous)
ax.text(8, 9.2, 'Physical Memory\n(non-contiguous)', ha='center', fontweight='bold', fontsize=10)
phys_colors = ['#e74c3c', '#f39c12', '#3498db', '#3498db', '#2ecc71']
for i in range(5):
    ax.add_patch(plt.Rectangle((7, 8-i*1.2), 2, 0.8, facecolor=phys_colors[i], alpha=0.6, edgecolor='black'))
    ax.text(8, 8.4-i*1.2, f'PPage {i}', ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(5.5, 8.4-[3,0,4,1,2].index(i)*1.2), xytext=(7, 8.4-i*1.2),
                arrowprops=dict(arrowstyle='->', color='gray', connectionstyle='arc3,rad=0.1'))

# Right: KV-Cache Paging
ax = axes[1]
ax.set_title('KV-Cache Paging (PagedAttention)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Logical KV blocks
for i in range(5):
    color = '#3498db'
    alpha = 0.6 if i < 4 else 0.2  # Last block partially filled
    ax.add_patch(plt.Rectangle((0.5, 8-i*1.2), 2, 0.8, facecolor=color, alpha=alpha, edgecolor='black'))
    tokens = '16 tokens' if i < 3 else ('10 tokens' if i == 3 else 'empty')
    ax.text(1.5, 8.4-i*1.2, f'LBlock {i}\n({tokens})', ha='center', va='center', fontsize=8)

ax.text(1.5, 9.2, 'Logical Blocks\n(request view)', ha='center', fontweight='bold', fontsize=10)

# Block table
ax.text(4.5, 9.2, 'Block Table', ha='center', fontweight='bold', fontsize=10)
phys_map = [7, 2, 12, 5, None]
for i in range(5):
    if phys_map[i] is not None:
        ax.add_patch(plt.Rectangle((3.5, 8-i*1.2), 2, 0.8, facecolor='lightyellow', edgecolor='black'))
        ax.text(4.5, 8.4-i*1.2, f'L{i} -> GPU[{phys_map[i]}]', ha='center', va='center', fontsize=8)
    else:
        ax.add_patch(plt.Rectangle((3.5, 8-i*1.2), 2, 0.8, facecolor='lightgray', alpha=0.3, edgecolor='black'))
        ax.text(4.5, 8.4-i*1.2, 'Not yet\nallocated', ha='center', va='center', fontsize=8, color='gray')

# GPU Memory blocks
ax.text(8, 9.2, 'GPU Memory\n(KV blocks)', ha='center', fontweight='bold', fontsize=10)
gpu_blocks = ['free', 'Req B[0]', 'Req A[1]', 'free', 'Req B[1]', 'Req A[3]', 'free', 'Req A[0]',
              'Req C[0]', 'free', 'free', 'Req C[1]', 'Req A[2]', 'free', 'Req B[2]']
gpu_colors_map = {'Req A': '#3498db', 'Req B': '#e74c3c', 'Req C': '#2ecc71', 'free': 'lightgray'}

for i in range(min(8, len(gpu_blocks))):
    key = 'free' if gpu_blocks[i] == 'free' else gpu_blocks[i][:5]
    c = gpu_colors_map.get(key, 'lightgray')
    ax.add_patch(plt.Rectangle((7, 8-i*1.0), 2, 0.7, facecolor=c, 
                                alpha=0.6 if c != 'lightgray' else 0.2, edgecolor='black'))
    ax.text(8, 8.35-i*1.0, f'[{i}] {gpu_blocks[i]}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 4. Block-Based Memory Management Implementation

Let's implement a complete KV-cache block allocator, similar to what vLLM uses internally.

In [ ]:
@dataclass
class KVBlock:
    """A single block of KV-cache memory."""
    block_id: int
    block_size: int  # number of token slots
    num_filled: int = 0  # how many token slots are used
    ref_count: int = 0   # reference count for CoW
    
    @property
    def is_full(self) -> bool:
        return self.num_filled >= self.block_size
    
    @property
    def free_slots(self) -> int:
        return self.block_size - self.num_filled


class BlockAllocator:
    """Manages physical KV-cache blocks on a device (GPU or CPU)."""
    
    def __init__(self, num_blocks: int, block_size: int, device: str = "gpu"):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.device = device
        
        # Initialize all blocks as free
        self.all_blocks = [KVBlock(block_id=i, block_size=block_size) 
                           for i in range(num_blocks)]
        self.free_blocks: List[int] = list(range(num_blocks))
        
        # Statistics
        self.total_allocations = 0
        self.total_frees = 0
    
    @property
    def num_free(self) -> int:
        return len(self.free_blocks)
    
    @property
    def utilization(self) -> float:
        return 1 - (self.num_free / self.num_blocks)
    
    def allocate(self) -> Optional[int]:
        """Allocate a single free block. Returns block_id or None if full."""
        if not self.free_blocks:
            return None
        
        block_id = self.free_blocks.pop(0)
        block = self.all_blocks[block_id]
        block.ref_count = 1
        block.num_filled = 0
        self.total_allocations += 1
        return block_id
    
    def free(self, block_id: int):
        """Free a block (decrement ref count, actually free if ref_count == 0)."""
        block = self.all_blocks[block_id]
        block.ref_count -= 1
        if block.ref_count <= 0:
            block.ref_count = 0
            block.num_filled = 0
            self.free_blocks.append(block_id)
            self.total_frees += 1
    
    def add_ref(self, block_id: int):
        """Add a reference to a block (for CoW)."""
        self.all_blocks[block_id].ref_count += 1
    
    def get_block(self, block_id: int) -> KVBlock:
        return self.all_blocks[block_id]


class BlockTable:
    """Maps logical blocks to physical blocks for a single request."""
    
    def __init__(self, request_id: str):
        self.request_id = request_id
        self.physical_blocks: List[int] = []  # List of physical block IDs
    
    def append_block(self, block_id: int):
        self.physical_blocks.append(block_id)
    
    @property
    def num_blocks(self) -> int:
        return len(self.physical_blocks)
    
    def get_physical_block(self, logical_idx: int) -> int:
        return self.physical_blocks[logical_idx]


class KVCacheManager:
    """High-level KV-cache manager that handles allocation for requests."""
    
    def __init__(self, num_gpu_blocks: int, num_cpu_blocks: int, block_size: int):
        self.block_size = block_size
        self.gpu_allocator = BlockAllocator(num_gpu_blocks, block_size, "gpu")
        self.cpu_allocator = BlockAllocator(num_cpu_blocks, block_size, "cpu")
        self.block_tables: Dict[str, BlockTable] = {}
    
    def allocate_request(self, request_id: str, num_tokens: int) -> bool:
        """Allocate blocks for a new request's initial tokens."""
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        
        if num_blocks_needed > self.gpu_allocator.num_free:
            return False  # Not enough memory
        
        block_table = BlockTable(request_id)
        for i in range(num_blocks_needed):
            block_id = self.gpu_allocator.allocate()
            if block_id is None:
                # Rollback
                for bid in block_table.physical_blocks:
                    self.gpu_allocator.free(bid)
                return False
            
            # Set fill level
            block = self.gpu_allocator.get_block(block_id)
            remaining = num_tokens - i * self.block_size
            block.num_filled = min(self.block_size, remaining)
            block_table.append_block(block_id)
        
        self.block_tables[request_id] = block_table
        return True
    
    def append_token(self, request_id: str) -> bool:
        """Append space for one new token to a request's KV-cache."""
        block_table = self.block_tables[request_id]
        
        if block_table.num_blocks == 0:
            # Need first block
            block_id = self.gpu_allocator.allocate()
            if block_id is None:
                return False
            block_table.append_block(block_id)
        
        # Check if last block has space
        last_block_id = block_table.physical_blocks[-1]
        last_block = self.gpu_allocator.get_block(last_block_id)
        
        if last_block.is_full:
            # Need a new block
            new_block_id = self.gpu_allocator.allocate()
            if new_block_id is None:
                return False
            block_table.append_block(new_block_id)
            last_block = self.gpu_allocator.get_block(new_block_id)
        
        last_block.num_filled += 1
        return True
    
    def free_request(self, request_id: str):
        """Free all blocks allocated to a request."""
        if request_id not in self.block_tables:
            return
        
        block_table = self.block_tables.pop(request_id)
        for block_id in block_table.physical_blocks:
            self.gpu_allocator.free(block_id)
    
    def get_stats(self) -> dict:
        return {
            'gpu_free_blocks': self.gpu_allocator.num_free,
            'gpu_total_blocks': self.gpu_allocator.num_blocks,
            'gpu_utilization': self.gpu_allocator.utilization,
            'active_requests': len(self.block_tables),
            'total_allocated_blocks': sum(bt.num_blocks for bt in self.block_tables.values()),
        }

# Demo: KV-Cache Manager in action
manager = KVCacheManager(num_gpu_blocks=32, num_cpu_blocks=64, block_size=4)

print("=== KV-Cache Block Manager Demo ===")
print(f"Configuration: 32 GPU blocks, 4 tokens per block\n")

# Allocate for 3 requests
requests = [
    ('req_A', 10),  # 10 tokens -> 3 blocks
    ('req_B', 5),   # 5 tokens -> 2 blocks  
    ('req_C', 14),  # 14 tokens -> 4 blocks
]

for req_id, num_tokens in requests:
    success = manager.allocate_request(req_id, num_tokens)
    bt = manager.block_tables[req_id]
    print(f"Allocated {req_id}: {num_tokens} tokens -> {bt.num_blocks} blocks "
          f"(physical: {bt.physical_blocks}), success={success}")

print(f"\nStats: {manager.get_stats()}")

# Generate tokens for req_A
print(f"\n--- Generating tokens for req_A ---")
for i in range(6):
    success = manager.append_token('req_A')
    bt = manager.block_tables['req_A']
    print(f"  Token {i+1}: blocks={bt.physical_blocks}, success={success}")

# Free req_B
print(f"\n--- Freeing req_B ---")
manager.free_request('req_B')
print(f"Stats after free: {manager.get_stats()}")

## 5. Visualizing Memory States

Let's create a visual memory map showing how blocks are allocated and freed over time.

In [ ]:
def visualize_memory_map(manager: KVCacheManager, title: str = "GPU Memory Map"):
    """Visualize the current state of GPU memory blocks."""
    num_blocks = manager.gpu_allocator.num_blocks
    cols = min(16, num_blocks)
    rows = (num_blocks + cols - 1) // cols
    
    fig, ax = plt.subplots(figsize=(max(8, cols * 0.8), max(3, rows * 0.8)))
    
    # Color map for requests
    request_colors = {}
    color_palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', 
                     '#1abc9c', '#e67e22', '#34495e']
    for i, req_id in enumerate(manager.block_tables.keys()):
        request_colors[req_id] = color_palette[i % len(color_palette)]
    
    # Build block -> request mapping
    block_to_request = {}
    for req_id, bt in manager.block_tables.items():
        for block_id in bt.physical_blocks:
            block_to_request[block_id] = req_id
    
    for block_id in range(num_blocks):
        row = block_id // cols
        col = block_id % cols
        y = rows - 1 - row
        
        block = manager.gpu_allocator.get_block(block_id)
        
        if block_id in block_to_request:
            req_id = block_to_request[block_id]
            color = request_colors[req_id]
            alpha = 0.4 + 0.6 * (block.num_filled / block.block_size)
            ax.add_patch(plt.Rectangle((col, y), 0.9, 0.9, facecolor=color, 
                                        alpha=alpha, edgecolor='black', linewidth=1))
            ax.text(col + 0.45, y + 0.55, f'{block_id}', ha='center', va='center', 
                    fontsize=7, fontweight='bold')
            ax.text(col + 0.45, y + 0.25, f'{block.num_filled}/{block.block_size}', 
                    ha='center', va='center', fontsize=6)
            if block.ref_count > 1:
                ax.text(col + 0.8, y + 0.8, f'R{block.ref_count}', ha='center', 
                        va='center', fontsize=5, color='red', fontweight='bold')
        else:
            ax.add_patch(plt.Rectangle((col, y), 0.9, 0.9, facecolor='lightgray', 
                                        alpha=0.3, edgecolor='gray', linewidth=0.5))
            ax.text(col + 0.45, y + 0.45, f'{block_id}', ha='center', va='center', 
                    fontsize=7, color='gray')
    
    ax.set_xlim(-0.1, cols + 0.1)
    ax.set_ylim(-0.1, rows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    # Legend
    patches = [mpatches.Patch(color=c, label=r, alpha=0.7) 
               for r, c in request_colors.items()]
    patches.append(mpatches.Patch(color='lightgray', label='Free', alpha=0.3))
    ax.legend(handles=patches, loc='upper right', fontsize=8, ncol=2)
    
    plt.tight_layout()
    plt.show()

# Show memory map after allocations
visualize_memory_map(manager, "GPU Memory Map After Allocations")

## 6. Copy-on-Write for Parallel Sampling

When generating multiple outputs from the same prompt (beam search, best-of-N, parallel sampling), all sequences share the same prompt KV-cache. **Copy-on-Write (CoW)** avoids duplicating this shared data.

### How It Works

1. All sequences point to the **same physical blocks** for the shared prefix
2. The blocks have a **reference count** > 1
3. When a sequence needs to **modify** a shared block (append a different token), the block is **copied first**
4. This is exactly how Unix `fork()` works with page tables!

In [ ]:
class CoWKVCacheManager(KVCacheManager):
    """KV-Cache manager with Copy-on-Write support."""
    
    def fork_request(self, source_id: str, new_id: str) -> bool:
        """Fork a request's KV-cache for parallel sampling.
        The new request shares all blocks with the source (CoW)."""
        if source_id not in self.block_tables:
            return False
        
        source_bt = self.block_tables[source_id]
        new_bt = BlockTable(new_id)
        
        # Share all blocks (increment ref counts)
        for block_id in source_bt.physical_blocks:
            self.gpu_allocator.add_ref(block_id)
            new_bt.append_block(block_id)
        
        self.block_tables[new_id] = new_bt
        return True
    
    def cow_append_token(self, request_id: str) -> bool:
        """Append a token with Copy-on-Write semantics.
        If the last block is shared, copy it first."""
        block_table = self.block_tables[request_id]
        
        if block_table.num_blocks == 0:
            return self.append_token(request_id)
        
        last_block_id = block_table.physical_blocks[-1]
        last_block = self.gpu_allocator.get_block(last_block_id)
        
        if last_block.is_full:
            # Need a new block (no CoW needed)
            new_id = self.gpu_allocator.allocate()
            if new_id is None:
                return False
            block_table.append_block(new_id)
            self.gpu_allocator.get_block(new_id).num_filled = 1
            return True
        
        if last_block.ref_count > 1:
            # Copy-on-Write: this block is shared, need to copy it
            new_block_id = self.gpu_allocator.allocate()
            if new_block_id is None:
                return False
            
            # Copy data (conceptually)
            new_block = self.gpu_allocator.get_block(new_block_id)
            new_block.num_filled = last_block.num_filled + 1
            
            # Update references
            self.gpu_allocator.free(last_block_id)  # Decrement old ref
            block_table.physical_blocks[-1] = new_block_id
            
            print(f"    [CoW] Block {last_block_id} copied to {new_block_id} for {request_id}")
            return True
        else:
            # Not shared, can modify in place
            last_block.num_filled += 1
            return True

# Demo: Copy-on-Write for beam search
print("=== Copy-on-Write Demo (Beam Search) ===")
cow_manager = CoWKVCacheManager(num_gpu_blocks=32, num_cpu_blocks=32, block_size=4)

# Original request with prompt
cow_manager.allocate_request('beam_base', 8)  # 8 tokens, 2 blocks
print(f"Base request allocated: blocks={cow_manager.block_tables['beam_base'].physical_blocks}")

# Fork into 3 beams
for i in range(3):
    beam_id = f'beam_{i}'
    cow_manager.fork_request('beam_base', beam_id)
    bt = cow_manager.block_tables[beam_id]
    print(f"Forked {beam_id}: blocks={bt.physical_blocks} (shared with base)")

# Check ref counts
for bid in cow_manager.block_tables['beam_base'].physical_blocks:
    block = cow_manager.gpu_allocator.get_block(bid)
    print(f"  Block {bid}: ref_count={block.ref_count}")

print(f"\n--- Generating different tokens for each beam ---")
# Each beam generates a different token -> triggers CoW
for i in range(3):
    beam_id = f'beam_{i}'
    success = cow_manager.cow_append_token(beam_id)
    bt = cow_manager.block_tables[beam_id]
    print(f"  {beam_id} after append: blocks={bt.physical_blocks}")

print(f"\nMemory stats: {cow_manager.get_stats()}")
visualize_memory_map(cow_manager, "Memory After CoW Fork + Divergence")

## 7. Prefix Sharing (Prefix Caching)

Many requests share common prompt prefixes:
- System prompts (e.g., "You are a helpful assistant...")
- Few-shot examples
- RAG context that overlaps between queries

**Prefix sharing** reuses the KV-cache for shared prefixes across different requests, avoiding redundant prefill computation.

### How It Works

1. Hash the token sequence for each block
2. If a block with the same hash exists, reuse it (increment ref count)
3. New blocks are allocated only for the unique suffix

In [ ]:
class PrefixCachingKVManager(CoWKVCacheManager):
    """KV-Cache manager with prefix caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Hash table: token_hash -> physical_block_id
        self.prefix_cache: Dict[int, int] = {}
    
    def _hash_tokens(self, tokens: List[int]) -> int:
        """Hash a sequence of tokens."""
        return hash(tuple(tokens))
    
    def allocate_with_prefix(self, request_id: str, tokens: List[int]) -> dict:
        """Allocate blocks with prefix caching.
        Returns info about cache hits/misses."""
        block_table = BlockTable(request_id)
        cache_hits = 0
        cache_misses = 0
        
        # Process tokens block by block
        for start in range(0, len(tokens), self.block_size):
            end = min(start + self.block_size, len(tokens))
            block_tokens = tokens[start:end]
            
            # Only cache full blocks
            if len(block_tokens) == self.block_size:
                # Include all previous tokens in hash (prefix dependency)
                prefix_hash = self._hash_tokens(tokens[:end])
                
                if prefix_hash in self.prefix_cache:
                    # Cache hit! Reuse existing block
                    block_id = self.prefix_cache[prefix_hash]
                    self.gpu_allocator.add_ref(block_id)
                    block_table.append_block(block_id)
                    cache_hits += 1
                    continue
            
            # Cache miss: allocate new block
            block_id = self.gpu_allocator.allocate()
            if block_id is None:
                # Rollback
                for bid in block_table.physical_blocks:
                    self.gpu_allocator.free(bid)
                return {'success': False}
            
            block = self.gpu_allocator.get_block(block_id)
            block.num_filled = len(block_tokens)
            block_table.append_block(block_id)
            cache_misses += 1
            
            # Add to cache if full block
            if len(block_tokens) == self.block_size:
                prefix_hash = self._hash_tokens(tokens[:end])
                self.prefix_cache[prefix_hash] = block_id
        
        self.block_tables[request_id] = block_table
        return {
            'success': True,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'tokens_saved': cache_hits * self.block_size,
        }

# Demo: Prefix caching
print("=== Prefix Caching Demo ===")
prefix_mgr = PrefixCachingKVManager(num_gpu_blocks=64, num_cpu_blocks=32, block_size=4)

# Shared system prompt tokens
system_prompt = [10, 20, 30, 40, 50, 60, 70, 80]  # 8 tokens = 2 full blocks

# Three requests with the same system prompt but different user messages
requests = [
    ('req_1', system_prompt + [100, 101, 102, 103, 104]),  # 13 tokens
    ('req_2', system_prompt + [200, 201, 202]),             # 11 tokens
    ('req_3', system_prompt + [300, 301, 302, 303]),        # 12 tokens
]

for req_id, tokens in requests:
    result = prefix_mgr.allocate_with_prefix(req_id, tokens)
    bt = prefix_mgr.block_tables[req_id]
    print(f"{req_id}: {len(tokens)} tokens, blocks={bt.physical_blocks}, "
          f"cache_hits={result['cache_hits']}, cache_misses={result['cache_misses']}, "
          f"tokens_saved={result['tokens_saved']}")

# Verify sharing
print("\n--- Block sharing verification ---")
for bid in range(10):
    block = prefix_mgr.gpu_allocator.get_block(bid)
    if block.ref_count > 0:
        owners = [req_id for req_id, bt in prefix_mgr.block_tables.items() if bid in bt.physical_blocks]
        print(f"  Block {bid}: ref_count={block.ref_count}, fill={block.num_filled}/{block.block_size}, "
              f"used by: {owners}")

visualize_memory_map(prefix_mgr, "Memory with Prefix Sharing")

## 8. CPU Offloading (Swapping)

When a request is preempted, its KV-cache can be saved to CPU memory and restored later. This is the "swap" preemption strategy from Chapter 2.2.

In [ ]:
class SwappableKVManager(KVCacheManager):
    """KV-Cache manager with CPU swap support."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.swapped_tables: Dict[str, List[int]] = {}  # req_id -> CPU block IDs
    
    def swap_out(self, request_id: str) -> dict:
        """Swap request's KV-cache from GPU to CPU."""
        if request_id not in self.block_tables:
            return {'success': False, 'reason': 'Request not found'}
        
        gpu_bt = self.block_tables[request_id]
        cpu_blocks = []
        
        # Allocate CPU blocks
        for gpu_bid in gpu_bt.physical_blocks:
            cpu_bid = self.cpu_allocator.allocate()
            if cpu_bid is None:
                # Rollback CPU allocations
                for cb in cpu_blocks:
                    self.cpu_allocator.free(cb)
                return {'success': False, 'reason': 'CPU memory full'}
            
            gpu_block = self.gpu_allocator.get_block(gpu_bid)
            cpu_block = self.cpu_allocator.get_block(cpu_bid)
            cpu_block.num_filled = gpu_block.num_filled
            cpu_blocks.append(cpu_bid)
        
        # Free GPU blocks
        for gpu_bid in gpu_bt.physical_blocks:
            self.gpu_allocator.free(gpu_bid)
        
        # Save mapping
        self.swapped_tables[request_id] = cpu_blocks
        del self.block_tables[request_id]
        
        num_blocks = len(cpu_blocks)
        transfer_size = num_blocks * self.block_size * 512  # Simulated bytes per token
        transfer_time_ms = transfer_size / (32 * 1e9) * 1000  # PCIe 4.0
        
        return {
            'success': True,
            'blocks_swapped': num_blocks,
            'transfer_size_kb': transfer_size / 1024,
            'estimated_time_ms': transfer_time_ms,
        }
    
    def swap_in(self, request_id: str) -> dict:
        """Swap request's KV-cache from CPU back to GPU."""
        if request_id not in self.swapped_tables:
            return {'success': False, 'reason': 'Request not in swap space'}
        
        cpu_blocks = self.swapped_tables[request_id]
        gpu_bt = BlockTable(request_id)
        
        # Allocate GPU blocks
        for cpu_bid in cpu_blocks:
            gpu_bid = self.gpu_allocator.allocate()
            if gpu_bid is None:
                for bid in gpu_bt.physical_blocks:
                    self.gpu_allocator.free(bid)
                return {'success': False, 'reason': 'GPU memory full'}
            
            cpu_block = self.cpu_allocator.get_block(cpu_bid)
            gpu_block = self.gpu_allocator.get_block(gpu_bid)
            gpu_block.num_filled = cpu_block.num_filled
            gpu_bt.append_block(gpu_bid)
        
        # Free CPU blocks
        for cpu_bid in cpu_blocks:
            self.cpu_allocator.free(cpu_bid)
        
        del self.swapped_tables[request_id]
        self.block_tables[request_id] = gpu_bt
        
        return {'success': True, 'blocks_restored': len(cpu_blocks)}

# Demo: Swapping
print("=== CPU Swap Demo ===")
swap_mgr = SwappableKVManager(num_gpu_blocks=16, num_cpu_blocks=32, block_size=4)

# Fill GPU with requests
swap_mgr.allocate_request('A', 12)  # 3 blocks
swap_mgr.allocate_request('B', 16)  # 4 blocks
swap_mgr.allocate_request('C', 8)   # 2 blocks

print(f"Before swap: GPU utilization = {swap_mgr.gpu_allocator.utilization:.1%}")
print(f"  GPU free: {swap_mgr.gpu_allocator.num_free}/{swap_mgr.gpu_allocator.num_blocks}")

# Swap out request B (largest)
result = swap_mgr.swap_out('B')
print(f"\nSwap out B: {result}")
print(f"After swap out: GPU free: {swap_mgr.gpu_allocator.num_free}, "
      f"CPU used: {swap_mgr.cpu_allocator.num_blocks - swap_mgr.cpu_allocator.num_free}")

# New request D can now fit
swap_mgr.allocate_request('D', 14)  # 4 blocks
print(f"After allocating D: GPU free: {swap_mgr.gpu_allocator.num_free}")

# Later, swap B back in
swap_mgr.free_request('D')  # D completes
result = swap_mgr.swap_in('B')
print(f"\nSwap in B: {result}")
print(f"After swap in: GPU free: {swap_mgr.gpu_allocator.num_free}")

## 9. Fragmentation Analysis: Contiguous vs Paged

Let's quantitatively compare memory utilization with and without paging under a realistic workload.

In [ ]:
def simulate_memory_utilization(use_paging: bool, num_requests: int = 100, 
                                 total_memory: int = 1000, block_size: int = 16):
    """Simulate memory utilization over time."""
    np.random.seed(42)
    
    if use_paging:
        num_blocks = total_memory  # each block = 1 unit
        mgr = KVCacheManager(num_blocks, num_blocks, block_size)
    
    # Track metrics
    utilization_over_time = []
    waste_over_time = []
    rejected = 0
    active_requests = {}  # req_id -> (reserved, actual_used, remaining_steps)
    
    # Contiguous allocator state
    if not use_paging:
        memory = [None] * total_memory  # None = free
        allocations = {}  # req_id -> (start, size)
    
    for step in range(200):
        # Arrivals (Poisson-ish)
        if step < num_requests:
            max_tokens = int(np.random.exponential(100)) + 20
            actual_gen = int(np.random.exponential(40)) + 5  # Usually much less than max
            actual_gen = min(actual_gen, max_tokens)
            prompt_tokens = int(np.random.exponential(50)) + 10
            total_reserved = prompt_tokens + max_tokens
            total_actual = prompt_tokens + actual_gen
            req_id = f'r{step}'
            
            if use_paging:
                blocks_needed = (prompt_tokens + block_size - 1) // block_size
                if blocks_needed <= mgr.gpu_allocator.num_free:
                    mgr.allocate_request(req_id, prompt_tokens)
                    active_requests[req_id] = {
                        'remaining': actual_gen,
                        'total_actual': total_actual
                    }
                else:
                    rejected += 1
            else:
                # Contiguous: must reserve for max_tokens
                # Find contiguous free space
                found = False
                for start in range(total_memory - total_reserved + 1):
                    if all(memory[j] is None for j in range(start, start + total_reserved)):
                        for j in range(start, start + total_reserved):
                            memory[j] = req_id
                        allocations[req_id] = (start, total_reserved)
                        active_requests[req_id] = {
                            'remaining': actual_gen,
                            'total_actual': total_actual,
                            'total_reserved': total_reserved
                        }
                        found = True
                        break
                if not found:
                    rejected += 1
        
        # Simulate generation
        to_remove = []
        for req_id, info in active_requests.items():
            if use_paging:
                mgr.append_token(req_id)
            info['remaining'] -= 1
            if info['remaining'] <= 0:
                to_remove.append(req_id)
        
        for req_id in to_remove:
            if use_paging:
                mgr.free_request(req_id)
            else:
                start, size = allocations.pop(req_id)
                for j in range(start, start + size):
                    memory[j] = None
            del active_requests[req_id]
        
        # Calculate utilization
        if use_paging:
            used = total_memory - mgr.gpu_allocator.num_free
            actual_needed = sum(
                sum(1 for bid in mgr.block_tables[rid].physical_blocks 
                    if mgr.gpu_allocator.get_block(bid).num_filled > 0)
                for rid in mgr.block_tables
            )
            waste = used - actual_needed
        else:
            used = sum(1 for m in memory if m is not None)
            actual_needed = sum(info.get('total_actual', 0) - info['remaining'] 
                               for info in active_requests.values())
            waste = used - min(actual_needed, used)
        
        utilization_over_time.append(used / total_memory)
        waste_over_time.append(waste / total_memory)
    
    return utilization_over_time, waste_over_time, rejected

# Run simulations
paged_util, paged_waste, paged_rejected = simulate_memory_utilization(True)
contig_util, contig_waste, contig_rejected = simulate_memory_utilization(False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(paged_util, label='Paged (total used)', color='#2ecc71', linewidth=2)
ax.plot(contig_util, label='Contiguous (total used)', color='#e74c3c', linewidth=2)
ax.fill_between(range(len(paged_waste)), 0, paged_waste, alpha=0.2, color='#2ecc71', label='Paged waste')
ax.fill_between(range(len(contig_waste)), 0, contig_waste, alpha=0.2, color='#e74c3c', label='Contiguous waste')
ax.set_xlabel('Time Step')
ax.set_ylabel('Memory Utilization')
ax.set_title('Memory Utilization: Paged vs Contiguous', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)

ax = axes[1]
avg_waste = [np.mean(paged_waste), np.mean(contig_waste)]
bars = ax.bar(['Paged', 'Contiguous'], avg_waste, color=['#2ecc71', '#e74c3c'], alpha=0.8)
ax.set_ylabel('Average Memory Waste')
ax.set_title(f'Average Waste & Rejection Rate', fontweight='bold')
ax.text(0, avg_waste[0] + 0.01, f'{avg_waste[0]:.1%}\nRejected: {paged_rejected}', 
        ha='center', fontweight='bold')
ax.text(1, avg_waste[1] + 0.01, f'{avg_waste[1]:.1%}\nRejected: {contig_rejected}', 
        ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Exercises

### Exercise 1: Build a Block Allocator with LRU Eviction

Extend the block allocator to support LRU (Least Recently Used) eviction for prefix cache entries.

In [ ]:
# Exercise 1: LRU Eviction for Prefix Cache

from collections import OrderedDict

class LRUPrefixCache:
    """YOUR CODE: Implement an LRU cache for prefix blocks.
    
    When GPU memory is full:
    1. Find the least recently used cached prefix block
    2. Evict it (free the GPU block)
    3. Allocate the freed space for the new request
    
    Track access time for each cached prefix.
    Only evict blocks with ref_count == 1 (not shared).
    """
    
    def __init__(self, max_cache_blocks: int = 100):
        self.max_cache_blocks = max_cache_blocks
        self.cache = OrderedDict()  # hash -> block_id
    
    def get(self, prefix_hash: int) -> Optional[int]:
        """YOUR CODE: Look up and return cached block, update access order."""
        pass
    
    def put(self, prefix_hash: int, block_id: int):
        """YOUR CODE: Cache a block, evict LRU if necessary."""
        pass
    
    def evict_lru(self) -> Optional[int]:
        """YOUR CODE: Evict the least recently used block."""
        pass

print("Exercise 1: Implement LRUPrefixCache and integrate it with PrefixCachingKVManager.")
print("Test with a workload where cache size is limited and eviction is needed.")

### Exercise 2: Memory Waste Analysis

Analyze the impact of block size on internal fragmentation. Smaller blocks = less waste per block, but more metadata overhead.

In [ ]:
# Exercise 2: Analyze block size impact

def analyze_block_size_tradeoff(block_sizes=[1, 4, 8, 16, 32, 64, 128], 
                                 num_requests=1000):
    """YOUR CODE: For each block size, simulate allocations and measure:
    1. Internal fragmentation (wasted space within blocks)
    2. Metadata overhead (number of block table entries)
    3. Total effective memory utilization
    
    Generate random requests with varying sequence lengths.
    Plot the results.
    """
    pass

print("Exercise 2: Implement analyze_block_size_tradeoff()")
print("Plot: Block Size vs Internal Fragmentation, Metadata Overhead, Total Utilization")
print("\nHint: Internal fragmentation per request = block_size - (total_tokens % block_size)")
print("Hint: vLLM uses block_size=16 as a good balance")

### Exercise 3: Implement a Radix-Tree Based Prefix Cache

SGLang uses a Radix tree for prefix caching, which is more efficient than hash-based lookup for hierarchical prefixes.

In [ ]:
# Exercise 3: Radix Tree Prefix Cache

class RadixTreeNode:
    """A node in the Radix tree for prefix caching."""
    def __init__(self):
        self.children: Dict[int, 'RadixTreeNode'] = {}  # token_id -> child
        self.block_id: Optional[int] = None  # Physical block if this completes a block
        self.ref_count: int = 0
        self.last_access_time: float = 0.0

class RadixPrefixCache:
    """YOUR CODE: Implement a Radix tree based prefix cache.
    
    Advantages over hash-based:
    1. Naturally handles prefix hierarchy
    2. Can find longest matching prefix efficiently
    3. Can evict partial prefixes
    """
    
    def __init__(self):
        self.root = RadixTreeNode()
    
    def match_prefix(self, tokens: List[int]) -> Tuple[int, List[int]]:
        """YOUR CODE: Find the longest matching prefix.
        Returns (num_tokens_matched, list_of_block_ids)."""
        pass
    
    def insert(self, tokens: List[int], block_ids: List[int]):
        """YOUR CODE: Insert a sequence with its block IDs."""
        pass

print("Exercise 3: Implement RadixPrefixCache.")
print("This is how SGLang's RadixAttention works internally.")

## 11. Summary

In this chapter, we explored memory management for KV-cache, the largest and most dynamic memory consumer in LLM serving:

1. **The fragmentation problem**: Naive contiguous allocation wastes memory through internal and external fragmentation
2. **PagedAttention**: Applies OS virtual memory concepts to KV-cache, using block tables for non-contiguous allocation
3. **Block allocator**: Fixed-size blocks eliminate external fragmentation; only last-block internal fragmentation remains
4. **Copy-on-Write**: Enables efficient parallel sampling (beam search) by sharing prompt KV-cache blocks
5. **Prefix sharing**: Reuses KV-cache across requests with common prefixes, avoiding redundant prefill
6. **CPU offloading**: Swaps preempted request KV-cache to CPU memory for later restoration

### Key Numbers to Remember

- KV-cache per token for Llama-3-8B: ~64 KB
- KV-cache for 4K context: ~256 MB per request
- Block size of 16 tokens is a good balance between fragmentation and metadata overhead
- Prefix caching can save 50-90% of prefill compute for workloads with shared prefixes

## What's Next?

In **Chapter 2.4: Model Parallelism Strategies**, we'll learn how to spread models across multiple GPUs when a single GPU doesn't have enough memory, covering tensor parallelism, pipeline parallelism, and expert parallelism.